# Data Preparation

In [1]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

con = sqlite3.connect("../data/hockey.db")
q = lambda sql: pd.read_sql(sql, con)
sql_file = lambda name: q(open(f"../sql/{name}").read())

sns.set_theme(style="whitegrid")

## Data Integration

In this section we join tables so that each row carries information from more than one source. The rubric asks for two things, and we document both:

- **Matching entities between sources**: the key used to link two tables.
- **Conversion of formats**: turning raw fields into values a model can use.

### Coach profile

Query: [`sql/coach_profile.sql`](../sql/coach_profile.sql). One row per coach stint (`coachID`, `year`, `tmID`, `lgID`, `stint`), for **all leagues**, with the person's details from `Master`.

| Source | Target | Key | What we bring in |
|---|---|---|---|
| `Coaches` | `Master` | `coachID` | birth date, `playerID`, `firstNHL` |

- We use a **left join** with `Coaches` as the driving table, so every coaching stint is kept even if the person were missing from `Master`.
- The result has **1,741 rows**, the same as `Coaches`. Every coach finds a match, and no `coachID` appears twice in `Master`, so the join neither loses nor duplicates rows.
- We keep **all leagues** (NHL, WHA, NHA, PCHA, WCHL), not only the NHL, so that a coach's experience can later include seasons coached outside the NHL. `lgID` is kept so we can filter to the NHL when building targets, and to join with `Teams` later.

#### Age at season start (format conversion)

`Master` stores the birth date in three separate columns (`birthYear`, `birthMon`, `birthDay`). We convert them into one number: the coach's **age at the start of the season**.

- **Reference date:** 1 August of `year`. In this dataset `year` is the season's starting year (2005 = 2005-06), so 1 August is before the season begins, and the age is known at prediction time.
- **Birthday rule:** if the birthday falls after 1 August, the coach has not turned the new age yet, so we subtract one year.
- **Missing month or day:** we assume 1 August, the same as the reference date. The age then becomes simply `year − birthYear`, which is wrong by at most one year.
- **Missing year:** the age is left missing (`<NA>`). It is not imputed here; that is handled in the *Missing values* section.

In [2]:
import datetime

coaches = sql_file("coach_profile.sql")

def calculate_age(row):
    if pd.isna(row["birthYear"]):
        return pd.NA
    
    birth_year = int(row["birthYear"])
    birth_month = 8 if pd.isna(row["birthMon"]) else int(row["birthMon"])
    birth_day = 1 if pd.isna(row["birthDay"]) else int(row["birthDay"])
    
    birth_date = datetime.date(
        birth_year,
        birth_month,
        birth_day
    )

    reference_date = datetime.date(
        int(row["year"]),
        8,
        1
    )

    age = reference_date.year - birth_date.year

    if (birth_date.month, birth_date.day) > (
        reference_date.month,
        reference_date.day
    ):
        age -= 1

    return age

coaches["age"] = coaches.apply(calculate_age, axis=1)
coaches["age"] = coaches["age"].astype("Int64")

#### Former player (format conversion)

`Master` only tells us *whether* a coach has a player record. We turn that into two 0/1 features:

| Feature | Rule | Meaning |
|---|---|---|
| `former_player` | `playerID` is not null | played in any league in the data |
| `former_nhl_player` | `firstNHL` is not null | played in the NHL |

Here a null does **not** mean "unknown": it means the person never played. So these flags have no missing values.

After building the features we drop the raw columns (`birthYear`, `birthMon`, `birthDay`, `playerID`, `firstNHL`), which are not useful to a model as they are.

In [3]:
import numpy as np
coaches["former_player"] = np.where(
    coaches["playerID"].isna(), 0, 1
)
coaches["former_nhl_player"] = np.where(
    coaches["firstNHL"].isna(), 0, 1
)

In [4]:
df = coaches.drop(["birthYear", "birthMon", "birthDay", "playerID", "firstNHL"], axis = 1)

#### Checks on the coach profile

In [5]:
younger_coach = df["age"].min()
oldest_coach = df["age"].max()
print(f'The youngest coach as age: {younger_coach}. The Oldest coach as age: {oldest_coach}')

The youngest coach as age: 22. The Oldest coach as age: 74


In [6]:
no_age = df["age"].isnull().sum()
print(f'There is {no_age} missing values')

null_age = df[df["age"].isnull()]
grouped_no_age = null_age.groupby('lgID').size().reset_index(name="count").sort_values("count",ascending=False)

print(grouped_no_age)

There is 60 missing values
   lgID  count
1   NHL     25
0   NHA     19
3  WCHL     10
2  PCHA      3
4   WHA      3


In [7]:
total = df["former_player"].count()
former_player = (df["former_player"] == 1).sum()

percentage = former_player / total * 100

print(f'The percentage of coaches that were former players is {percentage:.2f}%')

total = df["former_nhl_player"].count()
former_nhl_player = (df["former_nhl_player"] == 1).sum()

percentage = former_nhl_player / total * 100

print(f'The percentage of coaches that were former players is {percentage:.2f}%')


The percentage of coaches that were former players is 65.54%
The percentage of coaches that were former players is 63.12%


**Results**

- **Age range: 22 to 74.** Both are real cases, not data errors. The youngest is a player-coach in the NHA in 1916, when teams often had a player also coaching. The oldest is Al Arbour, who came back for one game with the Islanders in 2007. The youngest coach in the NHL itself is 25 (1979). No age is impossible, so no value is removed (validity).
- **60 stints (3.4%) have no age**, from 27 coaches with no `birthYear`. They come mostly from the early leagues (NHA, PCHA, WCHL). The 25 NHL rows are all from before 1940, so this gap should almost disappear after the era selection. We re-check it there (completeness).
- **Missing day or month only:** 30 more stints have a birth year but no month or day; their age uses the approximation described above.
- **Former players:** 65.5% of coaching stints are by former players, and 63.1% by former NHL players. These percentages count **stints (rows)**, not distinct coaches, so coaches with long careers count more than once.

### Coach History

Career features for each coach, built from [`sql/coach_history.sql`](../sql/coach_history.sql): one row per coach per season (all leagues), with the season's games, wins, losses, ties and playoff games summed over all the coach's stints in that season.

**Rule: only seasons before *t*.** Predictions are made the day before season *t* starts, so a coach's history for season *t* may only use seasons up to *t-1*. Anything from season *t* itself (its wins, its playoff games) is not known yet and would be leakage.

| Source | Key | What we bring in |
|---|---|---|
| `Coaches` (aggregated per season) | `coachID` | games, wins, ties and playoff games of every earlier season |

#### Sort by coach and year

The running totals below are computed in row order, so each coach's seasons must be in chronological order first.

In [8]:
history = sql_file("coach_history.sql")

history = history.sort_values(by=["coachID", "year"], ascending=[True, True])

print(history.head())

     coachID  year lgID  n_stints     g     w     l     t  postg
0  abelsi01c  1952  NHL         1  70.0  27.0  28.0  15.0    7.0
1  abelsi01c  1953  NHL         1  70.0  12.0  51.0   7.0    0.0
2  abelsi01c  1957  NHL         1  33.0  16.0  12.0   5.0    4.0
3  abelsi01c  1958  NHL         1  70.0  25.0  37.0   8.0    0.0
4  abelsi01c  1959  NHL         1  70.0  26.0  29.0  15.0    6.0


#### How a tie counts

A tie counts as **half a win**: win% = (wins + ½·ties) / games. This matches how hockey standings value a tie (1 point out of 2) and is consistent with `pts_pct` in the team features. It is expressed as a fraction (0–1), the same scale as `pts_pct`.

In [9]:
history["win_pct"] = (history["w"] + 0.5 * history["t"])/history["g"]

print(history.head())

     coachID  year lgID  n_stints     g     w     l     t  postg   win_pct
0  abelsi01c  1952  NHL         1  70.0  27.0  28.0  15.0    7.0  0.492857
1  abelsi01c  1953  NHL         1  70.0  12.0  51.0   7.0    0.0  0.221429
2  abelsi01c  1957  NHL         1  33.0  16.0  12.0   5.0    4.0  0.560606
3  abelsi01c  1958  NHL         1  70.0  25.0  37.0   8.0    0.0  0.414286
4  abelsi01c  1959  NHL         1  70.0  26.0  29.0  15.0    6.0  0.478571


#### Running totals (up to and including season *t*)

For each coach, a cumulative sum over his seasons in chronological order of games, wins, ties, playoff games and number of seasons. The sum is computed **within each coach** (grouped by `coachID`), so one coach's career never carries over into the next coach's rows.

At this point the totals still **include** season *t*, so they cannot be used as features yet.

In [10]:
count_columns = ["g", "w", "t", "l", "postg", "seasons_before"]

history["seasons_before"] = 1

career_columns = [
    "games_before",
    "wins_before",
    "ties_before",
    "losses_before",
    "postgames_before",
    "seasons_before"
]

history[career_columns] = history.groupby("coachID")[count_columns].cumsum()

print(history.head())

     coachID  year lgID  n_stints     g     w     l     t  postg   win_pct  \
0  abelsi01c  1952  NHL         1  70.0  27.0  28.0  15.0    7.0  0.492857   
1  abelsi01c  1953  NHL         1  70.0  12.0  51.0   7.0    0.0  0.221429   
2  abelsi01c  1957  NHL         1  33.0  16.0  12.0   5.0    4.0  0.560606   
3  abelsi01c  1958  NHL         1  70.0  25.0  37.0   8.0    0.0  0.414286   
4  abelsi01c  1959  NHL         1  70.0  26.0  29.0  15.0    6.0  0.478571   

   seasons_before  games_before  wins_before  ties_before  losses_before  \
0               1          70.0         27.0         15.0           28.0   
1               2         140.0         39.0         22.0           79.0   
2               3         173.0         55.0         27.0           91.0   
3               4         243.0         80.0         35.0          128.0   
4               5         313.0        106.0         50.0          157.0   

   postgames_before  
0               7.0  
1               7.0  
2       

#### Move the totals to "before season *t*"

Each running total is shifted down by one season within the same coach, so the row for season *t* holds the totals **up to season *t-1***. Career totals do not need consecutive seasons: if a coach skipped years, the shifted value is still the sum of all his earlier seasons.

**First season of each coach (rookie coach).** After the shift there is nothing before it, but the meaning differs per feature:

| Feature | First season | Why |
|---|---|---|
| seasons, games, wins, ties, playoff games before *t* | **0** | the coach really had zero |
| `win_career_pct` | **missing** | 0 / 0 is undefined; a rookie has no record, which is not the same as a 0% record. Handled in *Missing values* |
| `is_rookie_coach` | **1** | flags the rows where the career win% is missing for this reason |

In [11]:
history[career_columns] = history.groupby("coachID")[career_columns].shift(1).fillna(0.0)



history["win_before_pct"] = (history["wins_before"] + 0.5 * history["ties_before"])/history["games_before"]
history["is_rookie_coach"] = np.where(history["games_before"] == 0.0, 1, 0)
print(history.head())

     coachID  year lgID  n_stints     g     w     l     t  postg   win_pct  \
0  abelsi01c  1952  NHL         1  70.0  27.0  28.0  15.0    7.0  0.492857   
1  abelsi01c  1953  NHL         1  70.0  12.0  51.0   7.0    0.0  0.221429   
2  abelsi01c  1957  NHL         1  33.0  16.0  12.0   5.0    4.0  0.560606   
3  abelsi01c  1958  NHL         1  70.0  25.0  37.0   8.0    0.0  0.414286   
4  abelsi01c  1959  NHL         1  70.0  26.0  29.0  15.0    6.0  0.478571   

   seasons_before  games_before  wins_before  ties_before  losses_before  \
0             0.0           0.0          0.0          0.0            0.0   
1             1.0          70.0         27.0         15.0           28.0   
2             2.0         140.0         39.0         22.0           79.0   
3             3.0         173.0         55.0         27.0           91.0   
4             4.0         243.0         80.0         35.0          128.0   

   postgames_before  win_before_pct  is_rookie_coach  
0               0.0

#### Clean up

We drop the season's own results (`g`, `w`, `l`, `t`, `postg`, `win_pct`). They describe season *t* itself, so keeping them as features would be leakage. They are still available from the SQL if needed for the targets.

In [12]:
history = history.drop(["g", "w", "l", "t", "postg", "win_pct", "n_stints"], axis = 1)

print(history.head())

     coachID  year lgID  seasons_before  games_before  wins_before  \
0  abelsi01c  1952  NHL             0.0           0.0          0.0   
1  abelsi01c  1953  NHL             1.0          70.0         27.0   
2  abelsi01c  1957  NHL             2.0         140.0         39.0   
3  abelsi01c  1958  NHL             3.0         173.0         55.0   
4  abelsi01c  1959  NHL             4.0         243.0         80.0   

   ties_before  losses_before  postgames_before  win_before_pct  \
0          0.0            0.0               0.0             NaN   
1         15.0           28.0               7.0        0.492857   
2         22.0           79.0               7.0        0.357143   
3         27.0           91.0              11.0        0.395954   
4         35.0          128.0              11.0        0.401235   

   is_rookie_coach  
0                1  
1                0  
2                0  
3                0  
4                0  


#### Result

One row per (coach, season) with the coach's career up to the previous season. It is not attached to teams yet: that happens when building the targets, where we choose which coach describes each team-season.

In [13]:
is_rookie = (history["is_rookie_coach"] == 1).sum()

print(f'There is {is_rookie} values')

print(history.head())

There is 384 values
     coachID  year lgID  seasons_before  games_before  wins_before  \
0  abelsi01c  1952  NHL             0.0           0.0          0.0   
1  abelsi01c  1953  NHL             1.0          70.0         27.0   
2  abelsi01c  1957  NHL             2.0         140.0         39.0   
3  abelsi01c  1958  NHL             3.0         173.0         55.0   
4  abelsi01c  1959  NHL             4.0         243.0         80.0   

   ties_before  losses_before  postgames_before  win_before_pct  \
0          0.0            0.0               0.0             NaN   
1         15.0           28.0               7.0        0.492857   
2         22.0           79.0               7.0        0.357143   
3         27.0           91.0              11.0        0.395954   
4         35.0          128.0              11.0        0.401235   

   is_rookie_coach  
0                1  
1                0  
2                0  
3                0  
4                0  


### Awards

Awards from [`sql/coach_award.sql`](../sql/coach_award.sql): one row per coach-season in which the coach won an award, with the number of awards. **All award types are counted** (Jack Adams, Baldwin, Schmertz, First/Second Team All-Star), consistent with keeping all leagues for coaching experience.

| Source | Target | Key | What we bring in |
|---|---|---|---|
| `AwardsCoaches` | coach history | (`coachID`, `year`) | number of awards won that season |

**Leakage rule:** an award for season *t* is given after season *t* ends, so it only counts from season *t+1*. We apply the same running total + shift within each coach as for the career features.

In [14]:
awards = sql_file("coach_award.sql")

print(awards.head())

      coachID  year  awards_won
0  adamsja01c  1936           1
1  adamsja01c  1942           1
2  adamsja01c  1944           1
3  arboual01c  1978           1
4  barbebi01c  2000           1


#### Merge

Left join onto the coach history, so every coach-season is kept. A season with no award has no row in `AwardsCoaches`, so after the join it is NULL; here NULL means "no award", not "unknown", so it becomes 0.

In [15]:
merged_history = pd.merge(history, awards, on=["coachID", "year"], how="left")

merged_history["awards_won"] = merged_history["awards_won"].fillna(0)
print(merged_history.head())

     coachID  year lgID  seasons_before  games_before  wins_before  \
0  abelsi01c  1952  NHL             0.0           0.0          0.0   
1  abelsi01c  1953  NHL             1.0          70.0         27.0   
2  abelsi01c  1957  NHL             2.0         140.0         39.0   
3  abelsi01c  1958  NHL             3.0         173.0         55.0   
4  abelsi01c  1959  NHL             4.0         243.0         80.0   

   ties_before  losses_before  postgames_before  win_before_pct  \
0          0.0            0.0               0.0             NaN   
1         15.0           28.0               7.0        0.492857   
2         22.0           79.0               7.0        0.357143   
3         27.0           91.0              11.0        0.395954   
4         35.0          128.0              11.0        0.401235   

   is_rookie_coach  awards_won  
0                1         0.0  
1                0         0.0  
2                0         0.0  
3                0         0.0  
4          

#### Awards Before

In [16]:
merged_history["awards_before"] = 0

merged_history["awards_before"] = merged_history.groupby("coachID")["awards_won"].cumsum()

merged_history["awards_before"] = merged_history.groupby("coachID")["awards_before"].shift(1).fillna(0)

print(merged_history.head())

     coachID  year lgID  seasons_before  games_before  wins_before  \
0  abelsi01c  1952  NHL             0.0           0.0          0.0   
1  abelsi01c  1953  NHL             1.0          70.0         27.0   
2  abelsi01c  1957  NHL             2.0         140.0         39.0   
3  abelsi01c  1958  NHL             3.0         173.0         55.0   
4  abelsi01c  1959  NHL             4.0         243.0         80.0   

   ties_before  losses_before  postgames_before  win_before_pct  \
0          0.0            0.0               0.0             NaN   
1         15.0           28.0               7.0        0.492857   
2         22.0           79.0               7.0        0.357143   
3         27.0           91.0              11.0        0.395954   
4         35.0          128.0              11.0        0.401235   

   is_rookie_coach  awards_won  awards_before  
0                1         0.0            0.0  
1                0         0.0            0.0  
2                0         0.0  

#### Checks on the awards

1. **Row count:** the left join must not add or remove coach-seasons.
2. **Awards lost in the join:** awards whose (coach, season) does not exist in `Coaches`. We expect exactly one: Dave Tippett, Jack Adams 2009, with no 2009 season in `Coaches` (data quality: accuracy). Losing it does not matter for the features, because there is no season after 2009 in which it could count.
3. **Rookie coaches** (first season) must have `awards_before == 0`. A positive value means a previous coach's awards leaked into the next coach's first row.
4. **Never goes down:** within a coach, `awards_before` can only stay the same or grow.
5. **Hand check:** Dick Irvin (`irvindi01c`, the most awarded coach). Each award in `awards_won` must appear one season later in `awards_before`.

In [17]:
# 1. row count unchanged by the join
print(f"1. rows: history = {len(history)}, merged = {len(merged_history)} ->",
      "OK" if len(history) == len(merged_history) else "FAIL")

# 2. awards lost in the join
lost = awards.merge(history[["coachID", "year"]], on=["coachID", "year"], how="left", indicator=True)
lost = lost[lost["_merge"] == "left_only"]
print(f"2. awards with no matching coach-season: {len(lost)}")
print(lost[["coachID", "year", "awards_won"]].to_string(index=False))

# 3. rookie coaches must have no awards before
rookies = merged_history[merged_history["is_rookie_coach"] == 1]
bad_rookies = rookies[rookies["awards_before"] > 0]
print(f"3. rookie coaches with awards_before > 0: {len(bad_rookies)} ->", "OK" if len(bad_rookies) == 0 else "FAIL")
print(bad_rookies[["coachID", "year", "awards_before"]].head().to_string(index=False))

# 4. awards_before never goes down within a coach
goes_down = merged_history.groupby("coachID")["awards_before"].diff() < 0
print(f"4. rows where awards_before goes down: {goes_down.sum()} ->", "OK" if goes_down.sum() == 0 else "FAIL")

# 5. hand check: Dick Irvin
print("5. Dick Irvin: each award should appear one season later in awards_before")
print(merged_history.loc[merged_history["coachID"] == "irvindi01c", ["year", "awards_won", "awards_before"]].head(12).to_string(index=False))

1. rows: history = 1729, merged = 1729 -> OK
2. awards with no matching coach-season: 1
   coachID  year  awards_won
tippeda01c  2009           1
3. rookie coaches with awards_before > 0: 0 -> OK
Empty DataFrame
Columns: [coachID, year, awards_before]
Index: []
4. rows where awards_before goes down: 0 -> OK
5. Dick Irvin: each award should appear one season later in awards_before
 year  awards_won  awards_before
 1928         0.0            0.0
 1930         1.0            0.0
 1931         1.0            1.0
 1932         1.0            2.0
 1933         1.0            3.0
 1934         1.0            4.0
 1935         0.0            5.0
 1936         0.0            5.0
 1937         0.0            5.0
 1938         0.0            5.0
 1939         0.0            5.0
 1940         1.0            5.0


#### Clean Up

In [18]:
merged_history = merged_history.drop(["awards_won"], axis = 1)

### Tenure

**Tenure** = number of **consecutive** seasons the coach has led **this franchise** before season *t*. Unlike the career features, it is counted per franchise and a gap breaks the streak (a coach who comes back after years away starts again from 0).

Query: [`sql/coach_tenure.sql`](../sql/coach_tenure.sql), one row per (coach, franchise, season).

| Source | Target | Key | What we bring in |
|---|---|---|---|
| `Coaches` | `Teams` | (`year`, `tmID`, `lgID`) | `franchID` |

We use `franchID`, not `tmID`, because teams relocate and change code (28 franchises used more than one `tmID`); a relocated team is still the same job. Two stints of the same coach with the same franchise in one season count as one season.

In [19]:
tenure = sql_file("coach_tenure.sql")

print(tenure.head())

      coachID  year lgID franchID
0   boondi01c  1909  NHA      MTW
1   hareto01c  1909  NHA      BKN
2  lavioja01c  1909  NHA      TBS
3  stuarbr02c  1909  NHA      STE
4  triheha01c  1909  NHA      MOS


#### Streaks

**The lockout.** There was no 2004 season, so a coach who led the same team in 2003 and 2005 never left. Comparing calendar years would wrongly start a new streak in 2005. Instead, each season gets a **season number** (its position among the seasons that exist in `Teams`), so 2003 and 2005 are consecutive, and any real gap is still detected.

**Counting the streak**, within each (coach, franchise), sorted by year:
1. Mark the rows that **start a new streak**: the first season with that franchise, or a season whose number is not the previous one + 1.
2. A **running count** of those marks gives each streak an id.
3. Within each streak, number the seasons from 0: that number is `tenure_before`, the seasons already completed with this franchise **before** season *t* (0 in the first season).

**Decisions.** A partial season (e.g. a coach hired mid-season, including the 9 interim stints) counts as a full season of tenure. The Detroit 1998 co-coaches each get the season.

In [20]:
tenure["season_number"] = (
    tenure["year"].ne(tenure["year"].shift()).cumsum()
)

print(tenure.head())

      coachID  year lgID franchID  season_number
0   boondi01c  1909  NHA      MTW              1
1   hareto01c  1909  NHA      BKN              1
2  lavioja01c  1909  NHA      TBS              1
3  stuarbr02c  1909  NHA      STE              1
4  triheha01c  1909  NHA      MOS              1


In [21]:
tenure = tenure.sort_values(["coachID", "franchID", "year"]).copy()

previous_season = tenure.groupby(["coachID", "franchID"])["season_number"].shift(1)
tenure["new_streak"] = previous_season.isna() | (tenure["season_number"] != previous_season + 1)

tenure["streak_id"] = tenure.groupby(["coachID", "franchID"])["new_streak"].cumsum()

tenure["tenure_before"] = tenure.groupby(["coachID", "franchID", "streak_id"]).cumcount()

print(tenure.head())

       coachID  year lgID franchID  season_number  new_streak  streak_id  \
369  abelsi01c  1952  NHL      CHI             44        True          1   
375  abelsi01c  1953  NHL      CHI             45       False          1   
401  abelsi01c  1957  NHL      DET             49        True          1   
409  abelsi01c  1958  NHL      DET             50       False          1   
416  abelsi01c  1959  NHL      DET             51       False          1   

     tenure_before  
369              0  
375              1  
401              0  
409              1  
416              2  


#### Checks on the tenure

1. **Row count:** one row per (coach, franchise, season). Expected **1,736**: `Coaches` has 1,741 stints, and in 5 cases a coach had two stints with the same franchise in the same season.
2. **Streak starts:** every streak starts at tenure 0, so the number of rows with `tenure_before == 0` must equal the number of distinct (coach, franchise) pairs plus the comebacks after a real gap. An independent count finds **72** comebacks (gaps other than the 2004 lockout).
3. **Lockout:** a coach who led the same franchise in 2003 and in 2005 has an unbroken streak (there was no 2004 season), so his 2005 tenure must be his 2003 tenure + 1, never 0. There are **22** such cases.
4. **Hand check:** Al Arbour (`arboual01c`) with the Islanders (`NYI`). His tenure must go back to 0 in **1988** and in **2007**, when he came back after years away.
5. **Consistency with the career table:** a coach cannot have been with one franchise longer than his whole career, so `tenure_before <= seasons_before` in every row.

In [22]:
# 1. row count
print(f"1. rows: {len(tenure)} (expected 1736) ->", "OK" if len(tenure) == 1736 else "FAIL")

# 2. number of streak starts = distinct (coach, franchise) pairs + 72 comebacks
n_pairs = tenure[["coachID", "franchID"]].drop_duplicates().shape[0]
n_starts = (tenure["tenure_before"] == 0).sum()
print(f"2. streak starts: {n_starts}, expected {n_pairs} pairs + 72 comebacks = {n_pairs + 72} ->",
      "OK" if n_starts == n_pairs + 72 else "FAIL")

# 3. lockout: 2003 -> 2005 with the same franchise must continue the streak
t03 = tenure[tenure["year"] == 2003][["coachID", "franchID", "tenure_before"]]
t05 = tenure[tenure["year"] == 2005][["coachID", "franchID", "tenure_before"]]
lockout = t03.merge(t05, on=["coachID", "franchID"], suffixes=("_2003", "_2005"))
broken = lockout[lockout["tenure_before_2005"] != lockout["tenure_before_2003"] + 1]
print(f"3. lockout cases: {len(lockout)}, streak wrongly broken in {len(broken)} ->",
      "OK" if len(broken) == 0 else "FAIL")
print(broken.head().to_string(index=False))

# 4. hand check: Al Arbour with the Islanders (reset expected in 1988 and 2007)
print("4. Al Arbour with NYI:")
print(tenure.loc[(tenure["coachID"] == "arboual01c") & (tenure["franchID"] == "NYI"),
                 ["year", "tenure_before"]].to_string(index=False))

# 5. tenure can never exceed the whole career
both = tenure.merge(merged_history[["coachID", "year", "seasons_before"]], on=["coachID", "year"], how="left")
too_long = both[both["tenure_before"] > both["seasons_before"]]
print(f"5. rows with tenure_before > seasons_before: {len(too_long)} ->", "OK" if len(too_long) == 0 else "FAIL")

1. rows: 1736 (expected 1736) -> OK
2. streak starts: 688, expected 616 pairs + 72 comebacks = 688 -> OK
3. lockout cases: 22, streak wrongly broken in 0 -> OK
Empty DataFrame
Columns: [coachID, franchID, tenure_before_2003, tenure_before_2005]
Index: []
4. Al Arbour with NYI:
 year  tenure_before
 1973              0
 1974              1
 1975              2
 1976              3
 1977              4
 1978              5
 1979              6
 1980              7
 1981              8
 1982              9
 1983             10
 1984             11
 1985             12
 1988              0
 1989              1
 1990              2
 1991              3
 1992              4
 1993              5
 2007              0
5. rows with tenure_before > seasons_before: 0 -> OK


#### Clean up

In [ ]:
tenure = tenure.drop(["season_number", "new_streak", "streak_id"], axis=1)
print(tenure.head())

       coachID  year lgID franchID  tenure_before
369  abelsi01c  1952  NHL      CHI              0
375  abelsi01c  1953  NHL      CHI              1
401  abelsi01c  1957  NHL      DET              0
409  abelsi01c  1958  NHL      DET              1
416  abelsi01c  1959  NHL      DET              2


## Data Quality

## Era Selection

## Targets

## Feature engineering

## Missing values

## Outliers

## Redundancy

## Save the Final table